# Chapter 7: Advanced ResponsesAgents and Custom Tools

This notebook demonstrates advanced techniques for developing and deploying intelligent agent and tool-integrated agents, focusing on MLflow's `ResponsesAgent` interface and Custom Tools. In this chapter, you will:

- Build and customize advanced `ResponsesAgent` models with MLflow
- Wrap and serve local or custom LLM providers through flexible agent interfaces
- Integrate external tools—like SQL functions, API calls, and custom data lookups—into tool-calling agent workflows
- Implement streaming responses, retain conversation history, and orchestrate multi-agents supervisor
- Package, version, and deploy generative AI agent applications using custom PyFuncs and MLflow Model Serving


## [To delete] Chapter 7: Advanced ChatAgents, Agents, and Custom PyFuncs
Learning Objective: Explore advanced development techniques for ResponsesAgent models and tool-integrated agents using MLflow.

 7.1 Developing Custom ResponsesAgent with MLflow
 - Why and When to Use Responses Agent
 - Wrapping Local LLM Providers
 - Dependencies, and Configuration Management


 7.2 Implementing Tool-Calling Models and Agents
 - Building Agents with Integrated Tools
 - Case Study: Extending the Chatbot with External Data Lookups
 - Examples: SQL Functions, API Calls, Vector Retrieval, Model Context Protocol (MCP) 
 - Tools Evaluations


 7.3 Integration with Advanced Capabilities
 - Streaming Responses
 - Conversation History and multi-turn conversations
     - Multi-Agent orchestration


 7.4 Packaging and Deploying Advanced GenAI applications as Custom PyFuncs
 - Deploying to Model Serving
 - Managing Dependencies and Model Versioning
 7.5 Conclusion & Best Practices


In [0]:
%pip install -U -qqqq backoff databricks-langchain langgraph==0.5.3 uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

## Agent tools
First, let's develop the tools that we will need. 

We'll need:
1. SQL Tool to retrieve customer bookings from Unity Airways booking data table. [structured data retrieval]
2. Vector Retriever tool for retrieving relevant question / answer from Unity Airways FAQ. [unstructured data retrieval]

For more examples of tools to add to your agent, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/agent-tool) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/agent-tool))


#### 1. SQL Tool for Customer Booking Retrieval

In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.unity_airways.lookup_customer_info(
  user_email STRING COMMENT 'Email of the customer whose info to look up.'
)
RETURNS TABLE
COMMENT 'Returns all metadata about a specific customer as a markdown table.'
RETURN (
  SELECT *
  FROM workspace.unity_airways.booking_records_dataset
  WHERE `primary_contact_email` LIKE user_email

)

##### Test the function
Test your function to check it works as expected. Specify a fully qualified function name in the execute_function API to run the function:

In [0]:
import pandas as pd 
from io import StringIO
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.lookup_customer_info",
  parameters={"user_email": 'alex.lim@example.com'}
)

pd.read_csv(StringIO(result.value))

#### 2. SQL Tool to Modify or Cancel Booking



In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.unity_airways.modify_cancel_booking(
  user_booking_id STRING COMMENT 'ID of the flight to change or cancel.',
  intent STRING COMMENT 'Intent of action: "modify" or "cancel"'
)
RETURNS TABLE(
  booking_id STRING,
  decision STRING COMMENT 'approved | denied | needs_agent', 
  status STRING,
  coupon_status STRING,
  refunded_amount DOUBLE,
  reason STRING COMMENT 'Explanation for the decision'
)
COMMENT 'Changes date/time or cancels booking, and gives back what decision is made: Decisions: approved | denied | needs_agent'
RETURN (
  WITH booking_info AS (
    SELECT 
      booking_id,
      status,
      policy_refundable,
      policy_changeable, 
      total_amount,
      refunded_amount,
      waiver_code,
      irrops_flag,
      schedule_change_flag,
      DATEDIFF(DAY, CURRENT_DATE(), travel_start_date) as days_until_travel
    FROM workspace.unity_airways.booking_records_dataset
    WHERE booking_id = user_booking_id
  ),
  
  decision_result AS (
    SELECT 
      bi.*,
      CASE 
        -- Deny invalid requests
        WHEN bi.booking_id IS NULL THEN 'denied'
        WHEN bi.status IN ('CANCELLED', 'REFUNDED') THEN 'denied' 
        WHEN intent NOT IN ('modify', 'cancel') THEN 'denied'
        WHEN intent = 'modify' AND bi.policy_changeable = FALSE AND bi.waiver_code IS NULL THEN 'denied'
        WHEN intent = 'cancel' AND bi.policy_refundable = FALSE AND bi.waiver_code IS NULL AND bi.irrops_flag = FALSE THEN 'denied'
        
        -- Auto approve with waivers or airline changes
        WHEN bi.waiver_code IS NOT NULL OR bi.irrops_flag = TRUE OR bi.schedule_change_flag = TRUE THEN 'approved'
        
        -- Modifications always need agent attention  
        WHEN intent = 'modify' THEN 'needs_agent'
        
        -- High value cancellations need agent approval (>$500)
        WHEN intent = 'cancel' AND bi.total_amount > 500 THEN 'needs_agent'
        
        -- Auto approve simple cancellations
        ELSE 'approved'
      END as decision,
      
      CASE 
        -- Denial reasons
        WHEN bi.booking_id IS NULL THEN 'Booking ID not found'
        WHEN bi.status IN ('CANCELLED', 'REFUNDED') THEN 'Booking already processed'
        WHEN intent NOT IN ('modify', 'cancel') THEN 'Invalid intent - must be "modify" or "cancel"'
        WHEN intent = 'modify' AND bi.policy_changeable = FALSE AND bi.waiver_code IS NULL THEN 'Ticket is non-changeable'
        WHEN intent = 'cancel' AND bi.policy_refundable = FALSE AND bi.waiver_code IS NULL AND bi.irrops_flag = FALSE THEN 'Ticket is non-refundable'
        
        -- Approval reasons
        WHEN bi.waiver_code IS NOT NULL THEN 'Approved with waiver code'
        WHEN bi.irrops_flag = TRUE THEN 'Approved due to airline operational disruption'
        WHEN bi.schedule_change_flag = TRUE THEN 'Approved due to schedule change'
        
        -- Agent review reasons
        WHEN intent = 'modify' THEN 'Modification requires agent review'
        WHEN intent = 'cancel' AND bi.total_amount > 500 THEN 'High-value cancellation requires agent approval'
        
        -- Default approval
        ELSE 'Standard cancellation approved'
      END as reason,
      
      CASE 
        WHEN intent = 'cancel' THEN
          CASE
            WHEN bi.waiver_code IS NOT NULL OR bi.irrops_flag = TRUE OR bi.schedule_change_flag = TRUE 
            THEN bi.total_amount  -- Full refund for waivers
            WHEN bi.policy_refundable = TRUE 
            THEN bi.total_amount * 0.8  -- 80% refund for refundable tickets
            ELSE 0.0  -- No refund for non-refundable
          END
        ELSE 0.0  -- No refund for modifications
      END as calculated_refund
    FROM booking_info bi
  )
  
  SELECT 
    dr.booking_id,
    dr.decision,
    CASE 
      WHEN dr.decision = 'approved' AND intent = 'cancel' THEN 'CANCELLED'
      WHEN dr.decision = 'approved' AND intent = 'modify' THEN 'MODIFIED'
      WHEN dr.decision = 'needs_agent' THEN 'PENDING_APPROVAL'
      ELSE dr.status
    END as status,
    CASE 
      WHEN dr.decision = 'approved' AND intent = 'cancel' THEN 'REFUNDED'
      WHEN dr.decision = 'approved' AND intent = 'modify' THEN 'EXCHANGED' 
      WHEN dr.decision = 'needs_agent' THEN 'PENDING'
      ELSE 'ORIGINAL'
    END as coupon_status,
    CASE 
      WHEN dr.decision = 'approved' AND intent = 'cancel' THEN dr.calculated_refund
      ELSE COALESCE(dr.refunded_amount, 0.0)
    END as refunded_amount,
    dr.reason
  FROM decision_result dr
);


In [0]:
import pandas as pd 
from io import StringIO
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

print("Test 1: This ticket is already refunded by the airline.")
client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.modify_cancel_booking",
  parameters={"user_booking_id": "xxc02dgtfgq5c34d",
              "intent": "cancel"}
)
display(pd.read_csv(StringIO(result.value)))

print("Test 2: This ticket fare is higher than 1000 and should be handled by an agent.")
client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.modify_cancel_booking",
  parameters={"user_booking_id": "cgvyie6ivwpvs7hz",
              "intent": "cancel"}
)
display(pd.read_csv(StringIO(result.value)))

print("Test 3: This ticket should be approved for refund.")
client = DatabricksFunctionClient()
result = client.execute_function(
  function_name="workspace.unity_airways.modify_cancel_booking",
  parameters={"user_booking_id": "c3dd03uvulemx15z",
              "intent": "cancel"}
)
display(pd.read_csv(StringIO(result.value)))

#### 3. Vector Retriever Tool for FAQ Retrieval

In [0]:
from databricks_langchain import VectorSearchRetrieverTool

# Use Databricks vector search index as tool
vector_retriever_tool = VectorSearchRetrieverTool(
                          index_name="workspace.unity_airways.faq_index",
                          num_results=5,
                          tool_description="Search through unity airways frequently asked question (FAQ) about flight cancellation, travel policies and baggages security."
                          )

##### Test the tool
Test the call to your retrieval tool. Specify a question and check if the results returned are coherent to the setting set on the retriever above.

In [0]:
results = vector_retriever_tool.invoke({
    "query": "Can my battery pack be transported in cabin baggages?"
})

So far, so good. Our two tools are working as expected.


## Define the agent in code
Define the agent code in a single cell below. This lets you easily write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.




### 1. Wrap the LangGraph agent using the `ResponsesAgent` interface

For compatibility with Databricks AI features, the `LangGraphResponsesAgent` class implements the `ResponsesAgent` interface to wrap the LangGraph agent.

Databricks recommends using `ResponsesAgent` as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's [ResponsesAgent documentation](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/).


In [0]:
%%writefile agent.py
import json
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict, Union
from uuid import uuid4

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    VectorSearchRetrieverTool,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
    convert_to_openai_messages,
)
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """You are an helpful flight assistant for Unity Airways, capable of supporting users with their bookings and policy questions using tools to retrieve information from Unity Airways knowledge bases.

Available user requests you can handle:
	•	Help customers find their flight reservations using booking reference (PNR), last name with email or phone, or ticket number.
	•	Assist in changing flight dates/times or canceling bookings, calculating and informing users about any applicable fees or waivers, and processing cancellations or changes.
	•	Answer customer questions about baggage allowances, change and refund rules, or disruption waivers by searching the policy FAQ.

You have access to these tools:
	•	lookup_booking: Given a PNR, ticket number, or name plus email/phone, retrieve full booking and policy context.
	•	modify_cancel_booking: Given booking context, change or cancel bookings, determine fees/waivers, and update booking state.
	•	faq_search: Given a natural language policy question, return the best answer, its source, and a confidence score.

When a customer asks a question:
	1.	Identify the primary intent: find booking, modify/cancel, or a policy query.
	2.	Gather necessary information (prompt the customer for any missing details).
	3.	Use the correct tool(s) to answer the request or perform the action.
	4.	For booking or change/cancel actions, clearly explain the outcome, including eligibility, rules, fees, refunds, or waivers.
	5.	If a policy query is made, search the FAQ and provide the most relevant, accurate information with attribution.
	6.	Escalate to a human agent if needed.
 
Always remain friendly, concise, and clear. Always confirm required information before taking any action. Make sure your responses are personalized and relevant to the user's intent. If the action asked cannot be handled, politely say so, and redirect to customer service contact email: support@unityairways.com. """

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## To create and see usage examples of more tools, see https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

# You can use UDFs in Unity Catalog as agent tools
UC_TOOL_NAMES = ["workspace.unity_airways.lookup_customer_info", "workspace.unity_airways.modify_cancel_booking"]
uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
tools.extend(uc_toolkit.tools)

#############################
## Vector Search for FAQ Tool
#############################
# Use Databricks vector search indexes as tools
VECTOR_SEARCH_TOOLS = []

VECTOR_SEARCH_TOOLS.append(
    VectorSearchRetrieverTool(
        index_name="workspace.unity_airways.faq_index",
        num_results=5,
        tool_description="Search through unity airways frequently asked question (FAQ) about flight cancellation, travel policies and baggages security."
    )
)
tools.extend(VECTOR_SEARCH_TOOLS)

#####################
## Define agent logic
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
):
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(AgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    def _responses_to_cc(self, message: dict[str, Any]) -> list[dict[str, Any]]:
        """Convert from a Responses API output item to ChatCompletion messages."""
        msg_type = message.get("type")
        if msg_type == "function_call":
            return [
                {
                    "role": "assistant",
                    "content": "tool call",
                    "tool_calls": [
                        {
                            "id": message["call_id"],
                            "type": "function",
                            "function": {
                                "arguments": message["arguments"],
                                "name": message["name"],
                            },
                        }
                    ],
                }
            ]
        elif msg_type == "message" and isinstance(message["content"], list):
            return [
                {"role": message["role"], "content": content["text"]}
                for content in message["content"]
            ]
        elif msg_type == "reasoning":
            return [{"role": "assistant", "content": json.dumps(message["summary"])}]
        elif msg_type == "function_call_output":
            return [
                {
                    "role": "tool",
                    "content": message["output"],
                    "tool_call_id": message["call_id"],
                }
            ]
        compatible_keys = ["role", "content", "name", "tool_calls", "tool_call_id"]
        filtered = {k: v for k, v in message.items() if k in compatible_keys}
        return [filtered] if filtered else []

    def _prep_msgs_for_cc_llm(self, responses_input) -> list[dict[str, Any]]:
        "Convert from Responses input items to ChatCompletion dictionaries"
        cc_msgs = []
        for msg in responses_input:
            cc_msgs.extend(self._responses_to_cc(msg.model_dump()))

    def _langchain_to_responses(self, messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
        "Convert from ChatCompletion dict to Responses output item dictionaries"
        for message in messages:
            message = message.model_dump()
            role = message["type"]
            if role == "ai":
                if tool_calls := message.get("tool_calls"):
                    return [
                        self.create_function_call_item(
                            id=message.get("id") or str(uuid4()),
                            call_id=tool_call["id"],
                            name=tool_call["name"],
                            arguments=json.dumps(tool_call["args"]),
                        )
                        for tool_call in tool_calls
                    ]
                else:
                    return [
                        self.create_text_output_item(
                            text=message["content"],
                            id=message.get("id") or str(uuid4()),
                        )
                    ]
            elif role == "tool":
                return [
                    self.create_function_call_output_item(
                        call_id=message["tool_call_id"],
                        output=message["content"],
                    )
                ]
            elif role == "user":
                return [message]

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = []
        for msg in request.input:
            cc_msgs.extend(self._responses_to_cc(msg.model_dump()))

        for event in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates", "messages"]):
            if event[0] == "updates":
                for node_data in event[1].values():
                    for item in self._langchain_to_responses(node_data["messages"]):
                        yield ResponsesAgentStreamEvent(type="response.output_item.done", item=item)
            # filter the streamed messages to just the generated text messages
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except Exception as e:
                    print(e)


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphResponsesAgent(agent)
mlflow.models.set_model(AGENT)

In [0]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

### 2. Test the agent

Interact with the agent to test its output and tool-calling abilities. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
result = AGENT.predict({"input": [{"role": "user", "content": "can the booking c3dd03uvulemx15z be cancelled?"}]})
result

In [0]:
for chunk in AGENT.predict_stream({"input": [{"role": "user", "content": "what's the policy for rescheduling my flight?"}]}):
    print(chunk.model_dump(exclude_none=True))

### 3. Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`

  - **TODO**: If your Unity Catalog tool queries a [vector search index](docs link) or leverages [external functions](docs link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).



In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
from agent import UC_TOOL_NAMES, VECTOR_SEARCH_TOOLS
import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution

resources = []
for tool in VECTOR_SEARCH_TOOLS:
    resources.extend(tool.resources)
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            "databricks-langchain",
            f"langgraph=={get_distribution('langgraph').version}",
            f"backoff=={get_distribution('backoff').version}",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
        ],
        resources=resources,
    )

## Evaluate the tools with Agent Evaluation

Use Mosaic AI Agent Evaluation to evaluate the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics.
See Databricks documentation ([AWS]((https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).


To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Can I bring my battery to cabin luggage?"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## Prepare for Deployment
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

### 1. Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Can my battery pack be transported in cabin baggages?"}]},
    env_manager="uv",
)

### 2. Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.


In [0]:
mlflow.set_registry_uri("databricks-uc")

catalog = "workspace"
schema = "unity_airways"
model_name = "unity-airways-booking-agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

### 3. Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=uc_registered_model_info.version,
    scale_to_zero=True
)

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See docs ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)) for details